# Engenharia de features

> **Autor**: Miguel Vieira Machado Pim  
> **Contexto**: Desafio do processo seletivo de estágio IndustriALL

Notebooks anteriores a este:

1. [`01_pre_processing`](01_pre_processing.ipynb)
2. [`02_data_visualization`](02_data_visualization.ipynb)

Neste notebook iremos agregar todas as outras informações dos notebooks anteriores para fazer o processo de engenharia de features a partir dos nossos dados.

## Bibliotecas

In [ ]:
import re
import numpy as np
import pandas as pd

from pathlib import Path

## Funções

In [ ]:
def preprocess_data(data_path: Path) -> pd.DataFrame:
    """
    Essa função coleta todos os 53 arquivos csv e converte eles em um único dataframe final.

    Args:
        data_path (Path): Caminho para a pasta que contém os arquivos csv.

    Returns:
        pd.DataFrame: Dataframe final contendo todas as features e a coluna target.
    """
    # Coletando todos os arquivos
    feature_dfs = []
    target_df = None

    for file in data_path.glob("*.csv"):
        if "target" in file.name:
            target_df = pd.read_csv(file)
            target_df.columns = ["timestamp", "target"]
        else:
            sensor_id = int(re.search(r"\d+", file.stem).group())
            
            df = pd.read_csv(file)
            df.columns = ["timestamp", f"sensor_{sensor_id}"]

            feature_dfs.append(df)
    
    feature_dfs.sort(
        key=lambda df: int(df.columns[1].split("_")[1])
    )
    
    # Tratando tipos dos datasets
    for df in feature_dfs:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    target_df["target"] = target_df["target"].map({
        "NORMAL": 0,
        "ANORMAL": 1
    }).astype(int)
    target_df["timestamp"] = pd.to_datetime(target_df["timestamp"])
    
    # Construindo dataset final
    df_final = (
        pd.concat(
            [target_df.set_index("timestamp")] +
            [df.set_index("timestamp") for df in feature_dfs],
            axis=1,
            join="outer" # Para mantermos timestamps que não estão em todas as tabelas
        )
        .sort_index()
    )
    
    return df_final

In [ ]:
def adicionar_features_temporais(
    df: pd.DataFrame,
    horizonte_horas: int,
) -> pd.DataFrame:
    """
    Cria o target de antecipação e as features temporais cíclicas.

    O target é positivo somente durante o período normal em que uma nova
    falha começará dentro do horizonte. Registros em que a máquina já está
    em falha ficam inelegíveis para treino e avaliação. As features cíclicas
    representam o horário atual, independentemente do horizonte.
    """
    if horizonte_horas <= 0:
        raise ValueError("O horizonte precisa ser positivo.")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("O índice precisa ser um DatetimeIndex.")

    df = df.sort_index().copy()
    estado_falha = df["target"].astype("int8").eq(1)
    inicios_falha = df.index[
        estado_falha
        & ~estado_falha.shift(fill_value=False)
    ]

    horizonte = pd.Timedelta(hours=horizonte_horas)
    target_antecipacao = pd.Series(
        0,
        index=df.index,
        dtype="Int8",
    )

    for inicio_falha in inicios_falha:
        janela_antecipacao = (
            (df.index >= inicio_falha - horizonte)
            & (df.index < inicio_falha)
            & ~estado_falha
        )
        target_antecipacao.loc[janela_antecipacao] = 1

    # Durante uma falha não há uma nova oportunidade de emitir alerta.
    target_antecipacao.loc[estado_falha] = pd.NA
    df[f"target_{horizonte_horas}h"] = target_antecipacao

    timestamp_atual = df.index

    # Hora com precisão de minutos
    minuto_do_dia = (
        timestamp_atual.hour * 60
        + timestamp_atual.minute
    )

    # Segunda-feira = 0 e domingo = 6
    dia_semana = timestamp_atual.dayofweek

    # Ciclo diário
    df["hora_sin"] = np.sin(
        2 * np.pi * minuto_do_dia / 1440
    )

    df["hora_cos"] = np.cos(
        2 * np.pi * minuto_do_dia / 1440
    )

    # Ciclo semanal
    df["dia_semana_sin"] = np.sin(
        2 * np.pi * dia_semana / 7
    )

    df["dia_semana_cos"] = np.cos(
        2 * np.pi * dia_semana / 7
    )

    return df

In [ ]:
def calcular_mad(values: np.ndarray) -> float:
    """
    Calcula o desvio absoluto mediano.

    MAD = mediana(|x - mediana(x)|)
    """
    mediana = np.nanmedian(values)

    return np.nanmedian(
        np.abs(values - mediana)
    )


def adicionar_features_agregadas_passadas(
    df: pd.DataFrame,
    sensores: list[str],
    sensores_anomalia: list[str],
    janelas: list[int],
    limite_anomalia: float = 4.0,
    remover_aquecimento: bool = False
) -> pd.DataFrame:
    """
    Cria features agregadas utilizando somente observações anteriores
    ao timestamp atual.

    Para cada sensor relacionado às falhas, calcula:
        - média;
        - mediana;
        - desvio-padrão;
        - MAD;
        - IQR;
        - quantidade de observações disponíveis.

    Para os sensores selecionados na análise de anomalias, calcula:
        - robust z-score;
        - indicador binário de anomalia.

    Também calcula tendências robustas comparando:
        - mediana de 10 min com mediana de 60 min;
        - mediana de 30 min com mediana de 120 min.

    Args:
        df:
            DataFrame ordenado e indexado por timestamp.

        sensores:
            Sensores relacionados aos dois tipos de falha.

        sensores_anomalia:
            Sensores selecionados pelo ranking final de anomalias.

        janelas:
            Tamanhos das janelas em número de registros.
            Para dados minuto a minuto, [10, 30, 60, 120]
            representa minutos.

        limite_anomalia:
            Limite absoluto do robust z-score para indicar anomalia.

        remover_aquecimento:
            Remove as primeiras linhas que ainda não possuem
            uma janela histórica completa.

    Returns:
        DataFrame original com as novas features.
    """
    df = df.sort_index().copy()

    sensores = list(dict.fromkeys(sensores))
    sensores_anomalia = list(dict.fromkeys(sensores_anomalia))
    sensores_calculo = list(dict.fromkeys([
        *sensores,
        *sensores_anomalia,
    ]))
    conjunto_sensores = set(sensores)
    conjunto_anomalias = set(sensores_anomalia)

    sensores_ausentes = set(sensores_calculo).difference(df.columns)
    if sensores_ausentes:
        raise ValueError(
            f"Sensores ausentes: {sorted(sensores_ausentes)}"
        )

    # Desloca os sensores em uma posição.
    # Assim, a feature da linha t utiliza no máximo o instante t - 1.
    historico = df[sensores_calculo].shift(1)

    novas_features = {}

    medianas_por_janela = {}
    mads_por_janela = {}

    for janela in janelas:

        rolling = historico.rolling(
            window=janela,
            min_periods=1
        )

        media = rolling.mean()
        mediana = rolling.median()

        # ddof=0 permite calcular a dispersão com uma observação.
        # Nesse caso, o desvio-padrão é legitimamente zero.
        desvio_padrao = rolling.std(ddof=0)

        q25 = rolling.quantile(0.25)
        q75 = rolling.quantile(0.75)

        mad = historico.rolling(
            window=janela,
            min_periods=1
        ).apply(
            calcular_mad,
            raw=True
        )

        quantidade_historico = rolling.count()

        medianas_por_janela[janela] = mediana
        mads_por_janela[janela] = mad

        for sensor in sensores_calculo:

            prefixo = f"{sensor}_roll_{janela}min"
            mediana_sensor = mediana[sensor]
            mad_sensor = mad[sensor]

            if sensor in conjunto_sensores:
                media_sensor = media[sensor]
                std_sensor = desvio_padrao[sensor]
                iqr_sensor = q75[sensor] - q25[sensor]

                novas_features[f"{prefixo}_mean"] = (
                    media_sensor
                )

                novas_features[f"{prefixo}_median"] = (
                    mediana_sensor
                )

                novas_features[f"{prefixo}_std"] = (
                    std_sensor
                )

                novas_features[f"{prefixo}_mad"] = (
                    mad_sensor
                )

                novas_features[f"{prefixo}_iqr"] = (
                    iqr_sensor
                )

                novas_features[f"{prefixo}_count"] = (
                    quantidade_historico[sensor]
                )

            if sensor in conjunto_anomalias:
                # O valor atual é comparado apenas com estatísticas passadas.
                # Quando MAD = 0, o robust z-score permanece NaN.
                denominador = mad_sensor.replace(0, np.nan)
                robust_z = (
                    0.6745
                    * (df[sensor] - mediana_sensor)
                    / denominador
                )

                novas_features[f"{prefixo}_robust_z"] = (
                    robust_z
                )

                novas_features[f"{prefixo}_is_anomaly"] = (
                    robust_z
                    .abs()
                    .gt(limite_anomalia)
                    .astype("int8")
                )

    # Tendência robusta entre uma janela curta e uma longa.
    if 10 in janelas and 60 in janelas:

        for sensor in sensores:

            denominador = (
                mads_por_janela[60][sensor]
                .replace(0, np.nan)
            )

            novas_features[
                f"{sensor}_trend_median_10_60"
            ] = (
                (
                    medianas_por_janela[10][sensor]
                    - medianas_por_janela[60][sensor]
                )
                / denominador
            )

    if 30 in janelas and 120 in janelas:

        for sensor in sensores:

            denominador = (
                mads_por_janela[120][sensor]
                .replace(0, np.nan)
            )

            novas_features[
                f"{sensor}_trend_median_30_120"
            ] = (
                (
                    medianas_por_janela[30][sensor]
                    - medianas_por_janela[120][sensor]
                )
                / denominador
            )

    features_df = pd.DataFrame(
        novas_features,
        index=df.index
    )

    resultado = pd.concat(
        [df, features_df],
        axis=1
    )

    if remover_aquecimento:
        resultado = resultado.iloc[max(janelas):].copy()

    return resultado

## Carregando os dados

In [ ]:
data_path = Path("../data")

industry_df = preprocess_data(data_path)

In [ ]:
feature_df = industry_df.copy()
feature_df

## Resumindo features que iremos adicionar

### Valores nulos

No notebook `03_eda_null_values` nós analisamos os valores nulos dos sensores e descobrimos que para alguns sensores, esses valores nulos explicam algumas das falhas. Os sensores escolhidos para adicionarmos uma feature se ele está nulo ou não foram:
- `sensor_51`, `sensor_0`, `sensor_6`, `sensor_7`, `sensor_8`, `sensor_9`, `sensor_50`

In [ ]:
# Sensores nos quais a ausência de valor pode conter informação
sensores_com_indicador = [
    "sensor_0",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_50",
    "sensor_51"
]

# Cria os indicadores
for sensor in sensores_com_indicador:
    feature_df[f"{sensor}_is_null"] = feature_df[sensor].isna().astype("int8")

A interpolação deverá ser feita posteriormente porque se for feita agora teremos **vazamento de dados**.

### Janelas dos sensores relacionados às falhas e anomalias

Para os sensores relacionados aos dois tipos de falha, iremos adicionar resumos das janelas anteriores. Para os sensores selecionados no gráfico final de anomalias, adicionaremos o robust z-score e um indicador binário usando o limite absoluto 4.

In [ ]:
SENSORES_FALHA_TIPO_1 = [
    "sensor_13",
    "sensor_12",
    "sensor_50",
    "sensor_11",
    "sensor_10",
    "sensor_48",
    "sensor_0",
]

SENSORES_FALHA_TIPO_2 = [
    "sensor_19",
    "sensor_28",
    "sensor_29",
    "sensor_23",
    "sensor_26",
    "sensor_21",
    "sensor_37",
    "sensor_30",
    "sensor_25",
    "sensor_20",
]

SENSORES_ANOMALIA = [
    "sensor_43",
    "sensor_46",
    "sensor_42",
    "sensor_39",
    "sensor_40",
    "sensor_41",
    "sensor_38",
    "sensor_45",
    "sensor_33",
    "sensor_11",
    "sensor_19",
    "sensor_35",
    "sensor_23",
    "sensor_25",
    "sensor_24",
]

SENSORES_PROMISSORES = list(dict.fromkeys([
    *SENSORES_FALHA_TIPO_1,
    *SENSORES_FALHA_TIPO_2,
]))

JANELAS = [10, 30, 60, 120]

feature_df = adicionar_features_agregadas_passadas(
    df=feature_df,
    sensores=SENSORES_PROMISSORES,
    sensores_anomalia=SENSORES_ANOMALIA,
    janelas=JANELAS,
    limite_anomalia=4.0,
    remover_aquecimento=False
)

### Tendência e Sazonalidade

Também iremos adicionar features relacionadas à hora atual e ao dia da semana, pois foi observado que essas informações possuem um nível de relação com as falhas. Essas features são independentes do horizonte de antecipação.

In [ ]:
HORIZONTES = [1, 2, 4, 6, 12, 24]

In [ ]:
feature_dfs = {}

for h in HORIZONTES:
    feature_dfs[h] = adicionar_features_temporais(
        feature_df,
        horizonte_horas=h
    )

Para cada horizonte, o target indica os períodos ainda normais em que uma nova falha começará dentro da janela de antecipação. Registros durante falhas permanecem inelegíveis. As features cíclicas representam sempre o horário atual do registro.

In [ ]:
feature_df

In [ ]:
features_path = Path("../feature_datasets")
features_path.mkdir(exist_ok=True, parents=True)

for h, df in feature_dfs.items():
    df.to_csv(
        features_path / f"feature_df_{h}h.csv",
        index=True,
        index_label="timestamp"
    )